1.1 环境配置与路径定义

首先，我们建立稳健的文件路径管理。我会自动创建一个 processed 文件夹来存放清洗后的数据，永远不要直接修改 input 文件夹里的原始文件（这是数据工程的第一原则）。

In [10]:
import pandas as pd
import os
from pathlib import Path

# ================= 配置区域 =================
# 1. 定义基础路径 (使用 pathlib 处理 Windows 路径更稳健)
BASE_DIR = Path(r"D:\wzy\Python\pyautogui-project\data")
INPUT_DIR = BASE_DIR / "input"
OUTPUT_DIR = BASE_DIR / "processed"

# 2. 确保输出目录存在
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 3. 定义文件名映射
FILES = {
    "hl": "hl_data.txt",
    "mwdl": "mwdl_data_sql_hl.txt",
    "yield": "spot_dwt_yield.txt"
}

# 设置 Pandas 显示选项 (方便调试)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

print(f"📂 输入目录: {INPUT_DIR}")
print(f"📂 输出目录: {OUTPUT_DIR}")
print("✅ 环境初始化完成")

📂 输入目录: D:\wzy\Python\pyautogui-project\data\input
📂 输出目录: D:\wzy\Python\pyautogui-project\data\processed
✅ 环境初始化完成


1.2 通用读取函数 (Robust Loader)

Spotfire 导出的文件有时会有编码问题或包含非标准的空值（如 "nan", "NULL"）。我们封装一个函数来统一处理。

In [8]:
def load_spotfire_tsv(filename):
    """
    读取 Spotfire 导出的 TSV (Tab-Separated Values) 文件
    自动处理编码和列名空格
    """
    file_path = INPUT_DIR / filename
    
    if not file_path.exists():
        print(f"⚠️ 文件不存在: {filename}")
        return None
    
    print(f"🔄 正在读取: {filename} ...")
    
    try:
        # Spotfire 导出通常是 Tab 分隔
        # encoding='utf-8' 是 IronPython StreamWriter 的默认行为
        # dtype=str 默认全读为字符串，防止 '007' 被读成 7，后续再手动转类型
        df = pd.read_csv(file_path, sep='\t', encoding='utf-8', dtype=str)
        
        # 清洗列名：去除前后空格，转为小写 (方便后续 coding)
        df.columns = df.columns.str.strip().str.lower()
        
        # 清洗数据：去除字符串两端的空格
        df = df.apply(lambda x: x.str.strip() if x.dtype == "object" else x)
        
        print(f"✅ 读取成功: {df.shape[0]} 行, {df.shape[1]} 列")
        return df
        
    except Exception as e:
        print(f"❌ 读取失败 {filename}: {e}")
        return None

# 测试读取
df_hl = load_spotfire_tsv(FILES["hl"])
df_mwdl = load_spotfire_tsv(FILES["mwdl"])
df_yield = load_spotfire_tsv(FILES["yield"])

🔄 正在读取: hl_data.txt ...
✅ 读取成功: 147 行, 41 列
🔄 正在读取: mwdl_data_sql_hl.txt ...
✅ 读取成功: 23243 行, 14 列
🔄 正在读取: spot_dwt_yield.txt ...
✅ 读取成功: 1021247 行, 23 列


1.3 预处理 HL 异常数据 (hl_data) / 预处理 MWDL数据 (mwdl_data_sql_hl)

In [13]:
if df_hl is not None:
    # 数值转换
    num_cols = ['ratio', 'ng_qty', 'input_qty', 'batch_gap', 'hl次数']
    for col in num_cols:
        if col in df_hl.columns:
            df_hl[col] = pd.to_numeric(df_hl[col], errors='coerce').fillna(0)
            
    # 过滤掉无效数据 (比如 input=0 的行)
    df_hl = df_hl[df_hl['input_qty'] > 0].copy()
    
    # 标记: 是否为新发异常 (HL次数 <= 1 且 恶化 > 0)
    df_hl['is_new_issue'] = (df_hl['hl次数'] <= 1) & (df_hl['batch_gap'] > 0)
    
    print("👉 HL数据清洗完毕，样例:")
    display(df_hl[['prod_code', 'defect_desc', 'batch_gap', 'is_new_issue']].head(3))

👉 HL数据清洗完毕，样例:


,prod_code,defect_desc,batch_gap,is_new_issue
0,C471,Aging AA区烧伤,0.00119,True
1,C471,单亮点,-0.00101,False
2,C471,单亮点,-0.00444,False


In [12]:
if df_mwdl is not None:
    # 1. 转换日期 (format根据实际数据调整，这里假设是 '20251120')
    if 'date_value' in df_mwdl.columns:
        df_mwdl['fab_date'] = pd.to_datetime(df_mwdl['date_value'], format='%Y%m%d', errors='coerce')
        
    # 2. 数值转换
    # 关键指标: ratio (良损率), ng_qty (不良数), input_qty (投入数)
    for col in ['ratio', 'ng_qty', 'input_qty', 'ratio_fanel']:
        if col in df_mwdl.columns:
            df_mwdl[col] = pd.to_numeric(df_mwdl[col], errors='coerce').fillna(0)

    print("👉 MWDL数据清洗完毕，Top 3 不良:")
    top_defects = df_mwdl.groupby('defect_desc')['ng_qty'].sum().nlargest(3)
    print(top_defects)

👉 MWDL数据清洗完毕，Top 3 不良:
defect_desc
S向亮线           371617
Aging AA区烧伤    196054
电流异常           184242
Name: ng_qty, dtype: int64


1.4 预处理panel明细数据 (spot_dwt_yield)

- 轻量化读取：只读取分析集中性所需的列，节省内存。

- 坐标清洗：确保 行 和 列 格式统一。

- 聚合分析：自动计算每个坐标点的不良集中度 (Heatmap Data)，而不是只傻傻地存明细。

In [ ]:
def process_panel_yield_data(df):
    if df is None:
        return None, {}
    
    print(f"🔄 正在处理良率明细数据，原始行数: {len(df)} ...")
    
    # ==========================================
    # 1. 数据清洗 (Data Cleaning)
    # ==========================================
    # 数值转换
    df['ng_qty'] = pd.to_numeric(df['ng_qty'], errors='coerce').fillna(0)
    
    # 提取不良记录 (只分析 NG 数据)
    df_ng = df[df['ng_qty'] > 0].copy()
    
    # 坐标清洗与合成 (行 + 列 -> 膜位)
    # 确保列存在且无空格
    if '行' in df_ng.columns and '列' in df_ng.columns:
        df_ng['pos_row'] = df_ng['行'].astype(str).str.strip()
        df_ng['pos_col'] = df_ng['列'].astype(str).str.strip()
        df_ng['membrane_pos'] = df_ng['pos_row'] + "-" + df_ng['pos_col']
    else:
        print("⚠️ 警告：未找到'行'或'列'字段，无法计算 Mapping 集中性")
        df_ng['membrane_pos'] = 'Unknown'

    print(f"✅ 清洗完成，不良记录数: {len(df_ng)}")

    # ==========================================
    # 2. 集中性分析 (Concentration Analysis)
    # ==========================================
    # 定义聚合层级：产品 -> 不良Code -> 批次(lot)
    # 注意：这里使用了 'lot' 字段作为批次维度
    group_keys = ['prod_code', 'defect_code', 'lot']
    
    # 确保 group_keys 在数据中存在
    missing_keys = [col for col in group_keys if col not in df_ng.columns]
    if missing_keys:
        print(f"❌ 错误：缺少关键聚合字段: {missing_keys}，请检查列名是否为 'lot' 或其他")
        return df_ng, {}

    def calculate_concentration(target_col):
        """内部函数：计算指定维度的集中性"""
        if target_col not in df_ng.columns:
            print(f"   ⚠️ 跳过 [{target_col}]：字段不存在")
            return pd.DataFrame()
            
        print(f"   ⚙️ 计算 [{target_col}] 集中性...")
        
        # A. 计算分子：每个 [产品-不良-批次] 下，该 Target 坏了多少个
        target_counts = df_ng.groupby(group_keys + [target_col]).size().reset_index(name='target_ng_count')
        
        # B. 计算分母：该 [产品-不良-批次] 的总不良数
        batch_totals = df_ng.groupby(group_keys).size().reset_index(name='batch_total_ng')
        
        # C. 关联并计算占比
        result = pd.merge(target_counts, batch_totals, on=group_keys, how='left')
        result['concentration_ratio'] = result['target_ng_count'] / result['batch_total_ng']
        
        # D. 排序：优先展示 数量多 且 占比高 的集中项
        result = result.sort_values(['target_ng_count', 'concentration_ratio'], ascending=[False, False])
        return result

    # 执行四维分析
    analysis_results = {
        "mapping": calculate_concentration('membrane_pos'),  # 膜位集中性
        "lot" : calculate_concentration('lot_id'),             # 批次集中性
        "sheet": calculate_concentration('sheet_id'),        # 大板集中性
        "glass": calculate_concentration('glass_id'),        # Glass集中性
        "sub_prod": calculate_concentration('sub_prod_id')   # 工单集中性
    }
    
    return df_ng, analysis_results

In [ ]:
# ==========================================
# 执行处理 (使用之前加载好的 df_yield)
# ==========================================
if 'df_yield' in locals() and df_yield is not None:
    df_panel_ng, concentration_results = process_panel_yield_data(df_yield)
    
    # 打印预览
    print("\n" + "="*30)
    print("🔥 Mapping 集中性 Top 3 (膜位):")
    if not concentration_results['mapping'].empty:
        display(concentration_results['mapping'].head(3))

    print("\n🔥 批次 集中性 Top 3 (工单):")
    if not concentration_results['lot'].empty:
        display(concentration_results['lot'].head(3))

    print("\n🔥 大板 集中性 Top 3 (大板):")
    if not concentration_results['sheet'].empty:
        display(concentration_results['sheet'].head(3))

    print("\n🔥 Glass 集中性 Top 3 (Glass):")
    if not concentration_results['glass'].empty:
        display(concentration_results['glass'].head(3))

    print("\n🔥 工单 集中性 Top 3 (工单):")
    if not concentration_results['sub_prod'].empty:
        display(concentration_results['sub_prod'].head(3))

🔄 正在处理良率明细数据，原始行数: 1021247 ...
✅ 清洗完成，不良记录数: 91091
   ⚙️ 计算 [membrane_pos] 集中性...
   ⚙️ 计算 [sheet_id] 集中性...
   ⚙️ 计算 [glass_id] 集中性...
   ⚙️ 计算 [sub_prod_id] 集中性...

🔥 Mapping 集中性 Top 3 (膜位):


,prod_code,defect_code,lot,membrane_pos,target_ng_count,batch_total_ng,concentration_ratio
29118,M678,DMLSM,25/11/04蒸镀批,1A-J0,493,620,0.795161
14901,M588,DMLSM,25/11/03蒸镀批,1A-J0,300,309,0.970874
1731,C471,JNCDN,25/10/12蒸镀批,1D-A0,236,869,0.271577


🔥 Sheet 集中性 Top 3 (大板):


,prod_code,defect_code,lot,sheet_id,target_ng_count,batch_total_ng,concentration_ratio
20298,M588,DFE00,25/11/03蒸镀批,L3MO5A01520,175,1041,0.168108
48925,M678,DMELA0,25/11/12蒸镀批,L3MR5A04E16,105,1831,0.057346
48920,M678,DMELA0,25/11/12蒸镀批,L3MR5A04E11,97,1831,0.052977
